# Q39: What methods can you use to evaluate the predictions in an Object Detection model?
**Topic:** Computer-vision | **Difficulty:** Easy | **Time:** 10 min
**Sheet:** Grind75ML

---

## Question
What methods can you use to evaluate the predictions in an Object Detection model?

## Detailed Answer

### 1. IoU (Intersection over Union)
The fundamental metric — measures overlap between predicted and ground truth boxes:

$$IoU = \frac{\text{Area of Intersection}}{\text{Area of Union}}$$

- IoU ≥ 0.5 → True Positive (common threshold)
- IoU < 0.5 → False Positive
- Undetected ground truth → False Negative

### 2. Precision & Recall
- **Precision** = TP / (TP + FP) → How many detections are correct?
- **Recall** = TP / (TP + FN) → How many objects were found?

### 3. Average Precision (AP)
Area under the Precision-Recall curve for a single class:

$$AP = \int_0^1 P(r) \, dr$$

Computed using the **11-point interpolation** (VOC) or **all-point interpolation** (COCO).

### 4. mAP (Mean Average Precision)
The **primary metric** for object detection — average AP across all classes:

$$mAP = \frac{1}{C} \sum_{i=1}^{C} AP_i$$

**COCO mAP variants:**
| Metric | IoU Threshold | Description |
|--------|--------------|-------------|
| mAP@0.5 | 0.50 | Lenient (PASCAL VOC standard) |
| mAP@0.75 | 0.75 | Strict localization |
| mAP@[.5:.95] | 0.50 to 0.95 (step 0.05) | **COCO primary metric** (average of 10 IoU thresholds) |

### 5. Size-Based Metrics (COCO)
| Metric | Object Size | Area |
|--------|------------|------|
| AP_small | < 32² pixels | < 1024 px² |
| AP_medium | 32²–96² pixels | 1024–9216 px² |
| AP_large | > 96² pixels | > 9216 px² |

### 6. Speed Metrics
| Metric | Description |
|--------|-----------|
| FPS | Frames per second |
| Latency | Time per image (ms) |
| FLOPs | Compute cost |

### 7. Additional Metrics
- **F1 Score**: Harmonic mean of precision and recall
- **GIoU / DIoU / CIoU**: Improved IoU variants for better loss computation
- **AR (Average Recall)**: Recall averaged over IoU thresholds
- **Log Average Miss Rate**: Used in pedestrian detection (Caltech benchmark)

In [ ]:
import numpy as np


def compute_iou(box1: np.ndarray, box2: np.ndarray) -> float:
    """Compute IoU between two boxes in [x1, y1, x2, y2] format."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    
    return inter / union if union > 0 else 0.0


def compute_ap(precisions: list[float], recalls: list[float]) -> float:
    """Compute Average Precision using all-point interpolation (COCO style)."""
    # Add sentinel values
    precisions = [1.0] + list(precisions) + [0.0]
    recalls = [0.0] + list(recalls) + [1.0]
    
    # Make precision monotonically decreasing
    for i in range(len(precisions) - 2, -1, -1):
        precisions[i] = max(precisions[i], precisions[i + 1])
    
    # Compute area under curve
    ap = 0.0
    for i in range(1, len(recalls)):
        if recalls[i] != recalls[i - 1]:
            ap += (recalls[i] - recalls[i - 1]) * precisions[i]
    return ap


def evaluate_detections(
    pred_boxes: list[np.ndarray],
    pred_scores: list[float],
    gt_boxes: list[np.ndarray],
    iou_threshold: float = 0.5
) -> dict:
    """Evaluate object detection predictions against ground truth."""
    # Sort predictions by confidence
    sorted_indices = np.argsort(pred_scores)[::-1]
    
    tp = np.zeros(len(pred_boxes))
    fp = np.zeros(len(pred_boxes))
    matched_gt = set()
    
    for i, idx in enumerate(sorted_indices):
        best_iou = 0.0
        best_gt = -1
        for j, gt_box in enumerate(gt_boxes):
            iou = compute_iou(pred_boxes[idx], gt_box)
            if iou > best_iou and j not in matched_gt:
                best_iou = iou
                best_gt = j
        
        if best_iou >= iou_threshold and best_gt >= 0:
            tp[i] = 1
            matched_gt.add(best_gt)
        else:
            fp[i] = 1
    
    cum_tp = np.cumsum(tp)
    cum_fp = np.cumsum(fp)
    precisions = cum_tp / (cum_tp + cum_fp)
    recalls = cum_tp / len(gt_boxes)
    
    ap = compute_ap(precisions.tolist(), recalls.tolist())
    
    return {
        'AP': ap,
        'Precision': precisions[-1] if len(precisions) > 0 else 0,
        'Recall': recalls[-1] if len(recalls) > 0 else 0,
        'TP': int(sum(tp)),
        'FP': int(sum(fp)),
        'FN': len(gt_boxes) - int(sum(tp))
    }


# --- Example ---
gt_boxes = [np.array([50, 50, 150, 150]), np.array([200, 200, 350, 350])]
pred_boxes = [np.array([55, 48, 155, 148]), np.array([210, 195, 345, 340]), np.array([400, 400, 500, 500])]
pred_scores = [0.9, 0.85, 0.7]

results = evaluate_detections(pred_boxes, pred_scores, gt_boxes)
print("=== Detection Evaluation ===")
for metric, value in results.items():
    print(f"  {metric}: {value:.4f}" if isinstance(value, float) else f"  {metric}: {value}")

## Key Takeaways
- **mAP@[.5:.95]** is the gold standard metric (COCO) — report this in papers/interviews
- Always report **size-based AP** to understand model performance on small/medium/large objects
- Consider the **speed-accuracy tradeoff** — mAP alone isn't enough for real-time applications
- Use **per-class AP** to identify which classes the model struggles with